In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap

In [ ]:
df = pd.read_csv('../../data/india_city_aqi_2015_2023-cleaned_dataset.csv')
df["aqi_category"] = df["aqi_category"].replace({
    "Poor": "Moderate"
})
df['aqi_category'].value_counts()

In [ ]:
df.head(10)

In [ ]:
# Features

features = [
    'pm2.5',
    'pm10',
    'no2',
    'so2',
    'co',
    'o3',
    'season'
 ]

X = df[features]
X = pd.get_dummies(X, columns=['season']) # season -> one hot encoded


In [ ]:
# Target encoding
from sklearn.preprocessing import LabelEncoder

y = df['aqi_category']
le = LabelEncoder()
y = le.fit_transform(y)

pd.Series(y).head(5)

In [ ]:
X.head(10)

In [ ]:
# Class label mapping (original -> encoded)
class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
class_mapping

In [ ]:
print(X.shape,y.shape)

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y

)

In [ ]:
# Training Random Forest Model

rf_model = RandomForestClassifier(

    n_estimators=200,      # number of trees
    max_depth=12,          # depth of each tree
    min_samples_split=5,   # minimum samples to split node
    min_samples_leaf=2,    # minimum samples in leaf node
    criterion="entropy",   # options: "gini", "entropy", "log_loss"
    random_state=42

)

rf_model.fit(X_train, y_train)

In [ ]:
# Making AQI Predictions category

y_pred_rf = rf_model.predict(X_test)

In [ ]:
# Evaluated Model's Peformance based on Accuracy

print("Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred_rf))



In [ ]:
# RF confusion matrix plot
labels = ["Good", "Moderate", "Satisfactory"]
cm = confusion_matrix(y_test, y_pred_rf, labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="viridis", colorbar=True, values_format="d")

plt.title("RF confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Feature Engineering -> Random forest provides automatic feature importance

importance = rf_model.feature_importances_
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importance
})
feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
    )
print(feature_importance)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance
)
plt.title("Feature Importance for AQI Prediction")
plt.show()